In [0]:
# ============================================================
# CELL 1: Imports and Configuration
# ============================================================
import uuid
import json
import hashlib
import traceback
from datetime import datetime
from pyspark.sql import functions as F, Window
from pyspark.sql.types import StringType, TimestampType, LongType, DoubleType, IntegerType
from delta.tables import DeltaTable

CATALOG        = "ubuntu_bank_fraud"
CONTROL_SCHEMA = "control"
SILVER_SCHEMA  = "silver"
BRONZE_SCHEMA  = "bronze"

BATCH_ID  = str(uuid.uuid4())
RUN_TS    = datetime.utcnow()

print(f"Batch ID  : {BATCH_ID}")
print(f"Run TS    : {RUN_TS}")
print(f"Catalog   : {CATALOG}")

Batch ID  : 51e149a9-09c3-48b8-9a2f-21ff34afed1e
Run TS    : 2026-06-23 12:29:01.466196
Catalog   : ubuntu_bank_fraud


/home/spark-e24d9109-d362-4fd7-bdd2-8f/.ipykernel/2061/command-6388236773342276-2659613669:19: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  RUN_TS    = datetime.utcnow()


In [0]:
# ============================================================
# CELL 2: Read active silver metadata rows
# ============================================================
metadata_df = spark.sql(f"""
    SELECT
        metadata_id,
        source_table,
        target_table,
        transformation_type,
        dq_rules,
        load_strategy,
        active_flag,
        execution_order,
        description
    FROM {CATALOG}.{CONTROL_SCHEMA}.silver_metadata
    WHERE active_flag = 1
    ORDER BY execution_order
""")

metadata_rows = metadata_df.collect()
print(f"Silver tables to process: {len(metadata_rows)}")
for r in metadata_rows:
    print(f"  [{r.execution_order}] {r.source_table} -> {r.target_table}  ({r.transformation_type})")

Silver tables to process: 7
  [1] bronze.brz_customers -> silver.silver_customers  (SCD2)
  [2] bronze.brz_accounts -> silver.silver_accounts  (SCD2)
  [3] bronze.brz_cards -> silver.silver_cards  (OVERWRITE)
  [4] bronze.brz_merchants -> silver.silver_merchants  (OVERWRITE)
  [5] bronze.brz_devices -> silver.silver_devices  (SCD2)
  [6] bronze.brz_customer_device_link -> silver.silver_customer_device_link  (OVERWRITE)
  [7] bronze.brz_transactions -> silver.silver_transaction_fact  (OVERWRITE)


In [0]:
# ============================================================
# CELL 3: Reusable audit and metrics writers
# ============================================================
def write_reject(bronze_table, reason, category, offending_value, record_id, bronze_batch_id=""):
    row = [(bronze_table, reason, category, str(offending_value), str(record_id), bronze_batch_id, datetime.utcnow())]
    df  = spark.createDataFrame(row, schema=["bronze_table","reject_reason","reject_category",
                                              "offending_value","record_identifier","bronze_batch_id","reject_ts"])
    df.write.format("delta").mode("append").saveAsTable(f"{CATALOG}.{CONTROL_SCHEMA}.silver_reject_log")


def write_duplicate_log(bronze_table, hash_col, dup_count):
    row = [(bronze_table, hash_col, int(dup_count), BATCH_ID, datetime.utcnow())]
    df  = spark.createDataFrame(row, schema=["bronze_table","record_hash","duplicate_count","batch_id","detected_ts"])
    df.write.format("delta").mode("append").saveAsTable(f"{CATALOG}.{CONTROL_SCHEMA}.silver_duplicate_log")


def write_dq_metrics(table_name, total, passed, rejected, duplicated):
    pass_rate = round((passed / total * 100), 2) if total > 0 else 0.0
    row = [(table_name, int(total), int(passed), int(rejected), int(duplicated), pass_rate, BATCH_ID, datetime.utcnow())]
    df  = spark.createDataFrame(row, schema=["table_name","total_rows","rows_passed","rows_rejected",
                                              "rows_duplicated","pass_rate_pct","batch_id","dq_check_ts"])
    df.write.format("delta").mode("append").saveAsTable(f"{CATALOG}.{CONTROL_SCHEMA}.silver_dq_metrics")

print("Audit writers registered.")

Audit writers registered.


In [0]:
# ============================================================
# CELL 4: Metadata-driven DQ Engine
# ============================================================
def run_dq_engine(df, source_table, dq_rules_json):
    """
    Applies DQ rules from JSON config.
    Returns: (clean_df, rejected_count)
    Rules supported: not_null, amount_min, dedup_key
    """
    if not dq_rules_json:
        return df, 0

    rules        = json.loads(dq_rules_json)
    total        = df.count()
    rejected     = 0
    clean_df     = df

    # ── NOT NULL checks ─────────────────────────────────────
    not_null_cols = rules.get("not_null", [])
    for col in not_null_cols:
        if col not in df.columns:
            continue
        null_df  = clean_df.filter(F.col(col).isNull())
        null_cnt = null_df.count()
        if null_cnt > 0:
            rejected += null_cnt
            # log each rejected record
            for row in null_df.limit(500).collect():
                rec_id = str(row[0]) if row[0] else "UNKNOWN"
                write_reject(source_table, f"NULL value in required column: {col}",
                             "NULL_VIOLATION", "NULL", rec_id, BATCH_ID)
            clean_df = clean_df.filter(F.col(col).isNotNull())
            print(f"  [DQ] NULL check on {col}: {null_cnt} rows rejected")

    # ── AMOUNT MIN check ────────────────────────────────────
    amount_min = rules.get("amount_min")
    if amount_min is not None and "amount" in clean_df.columns:
        bad_df  = clean_df.filter(F.col("amount") < amount_min)
        bad_cnt = bad_df.count()
        if bad_cnt > 0:
            rejected += bad_cnt
            for row in bad_df.limit(500).collect():
                write_reject(source_table, f"Amount below minimum: {amount_min}",
                             "INVALID_AMOUNT", str(row["amount"]), str(row[0]), BATCH_ID)
            clean_df = clean_df.filter(F.col("amount") >= amount_min)
            print(f"  [DQ] Amount min check: {bad_cnt} rows rejected")

    print(f"  [DQ] {source_table}: {total} total, {rejected} rejected, {total - rejected} clean")
    return clean_df, rejected


def remove_duplicates(df, source_table, dq_rules_json):
    """
    Removes duplicates using dedup_key from rules.
    Keeps row with latest _load_ts. Logs duplicate counts.
    """
    if not dq_rules_json:
        return df, 0

    rules    = json.loads(dq_rules_json)
    dedup_key = rules.get("dedup_key", "")
    if not dedup_key:
        return df, 0

    key_cols = [c.strip() for c in dedup_key.split(",")]
    # Validate all key cols exist
    key_cols = [c for c in key_cols if c in df.columns]
    if not key_cols:
        return df, 0

    total_before = df.count()

    # Add row hash for duplicate logging
    hash_expr = F.md5(F.concat_ws("|", *[F.col(c).cast("string") for c in key_cols]))
    df = df.withColumn("_row_hash", hash_expr)

    # Find duplicates
    dup_counts = (
        df.groupBy("_row_hash")
          .count()
          .filter(F.col("count") > 1)
    )
    dup_count = dup_counts.count()

    if dup_count > 0:
        for row in dup_counts.limit(200).collect():
            write_duplicate_log(source_table, row["_row_hash"], row["count"])

    # Keep latest record per key
    ts_col = "_load_ts" if "_load_ts" in df.columns else key_cols[0]
    window = Window.partitionBy(*key_cols).orderBy(F.col(ts_col).desc())
    deduped = (
        df.withColumn("_rn", F.row_number().over(window))
          .filter(F.col("_rn") == 1)
          .drop("_rn", "_row_hash")
    )

    total_after  = deduped.count()
    dupes_removed = total_before - total_after
    print(f"  [DEDUP] {source_table}: {dupes_removed} duplicates removed")
    return deduped, dupes_removed

print("DQ engine registered.")

DQ engine registered.


In [0]:
# ============================================================
# CELL 5: SCD Type 2 merge writer
# ============================================================
def apply_scd2(spark, new_df, target_full_name, key_col):
    """
    Implements SCD Type 2 using Delta MERGE.
    - Closes existing current records when business attributes change
    - Inserts new version as current
    - Passes through unchanged records
    """
    now = datetime.utcnow()

    # Add SCD2 columns to incoming data
    new_df = (
        new_df
        .withColumn("effective_start_date", F.lit(now).cast(TimestampType()))
        .withColumn("effective_end_date",   F.lit(None).cast(TimestampType()))
        .withColumn("is_current",           F.lit(1).cast(IntegerType()))
        .withColumn("_batch_id",            F.lit(BATCH_ID))
    )

    # Check if target table exists
    table_exists = spark.catalog.tableExists(target_full_name)

    if not table_exists:
        # First load — just write
        (
            new_df.write
                .format("delta")
                .mode("overwrite")
                .option("mergeSchema", "true")
                .saveAsTable(target_full_name)
        )
        print(f"  [SCD2] Initial load to {target_full_name}: {new_df.count()} rows")
        return

    # Non-SCD2 columns to compare (exclude audit cols)
    exclude_cols = {key_col, "effective_start_date", "effective_end_date",
                    "is_current", "_load_ts", "_batch_id", "_source_file"}
    compare_cols = [c for c in new_df.columns if c not in exclude_cols]

    # Build change detection expression
    change_expr = " OR ".join([
        f"target.{c} <> source.{c} OR (target.{c} IS NULL AND source.{c} IS NOT NULL)"
        for c in compare_cols if c in new_df.columns
    ])

    delta_target = DeltaTable.forName(spark, target_full_name)

    # Step 1: Close changed records
    delta_target.alias("target").merge(
        new_df.alias("source"),
        f"target.{key_col} = source.{key_col} AND target.is_current = 1"
    ).whenMatchedUpdate(
        condition=change_expr,
        set={
            "effective_end_date": F.lit(now).cast(TimestampType()),
            "is_current": F.lit(0)
        }
    ).execute()

    # Step 2: Insert new versions (only changed or new records)
    existing_current = spark.table(target_full_name).filter(F.col("is_current") == 1)

    # Find records that are new or changed
    unchanged_keys = (
        existing_current.alias("e")
        .join(new_df.alias("n"), key_col)
        .filter(
            " AND ".join([f"e.{c} = n.{c}" for c in compare_cols if c in new_df.columns])
        )
        .select(F.col(f"n.{key_col}"))
    )

    records_to_insert = new_df.join(unchanged_keys, key_col, "left_anti")
    insert_count = records_to_insert.count()

    if insert_count > 0:
        (
            records_to_insert.write
                .format("delta")
                .mode("append")
                .option("mergeSchema", "true")
                .saveAsTable(target_full_name)
        )

    print(f"  [SCD2] {target_full_name}: {insert_count} new/changed versions inserted")

print("SCD2 writer registered.")

SCD2 writer registered.


In [0]:
# ============================================================
# CELL 6: Reusable transformation functions
# ============================================================
def standardize_strings(df, cols):
    """Trim and upper-case string columns."""
    for c in cols:
        if c in df.columns:
            df = df.withColumn(c, F.upper(F.trim(F.col(c))))
    return df


def cast_numeric(df, cols, dtype=DoubleType()):
    """Safely cast columns to numeric, nullify non-parseable values."""
    for c in cols:
        if c in df.columns:
            df = df.withColumn(c, F.col(c).cast(dtype))
    return df


def add_silver_audit_cols(df):
    """Add standard Silver audit columns."""
    return (
        df
        .withColumn("_silver_load_ts", F.current_timestamp())
        .withColumn("_silver_batch_id", F.lit(BATCH_ID))
    )


def apply_transformation(df, source_table):
    """
    Route to table-specific transformations.
    All transformations are pure column-level operations —
    no hardcoded business logic that cannot be generalised.
    """
    name = source_table.split(".")[-1].replace("brz_", "")

    if name == "customers":
        df = standardize_strings(df, ["first_name", "last_name", "email", "country", "city"])
        df = df.withColumn("email", F.lower(F.trim(F.col("email")))) if "email" in df.columns else df

    elif name == "accounts":
        df = standardize_strings(df, ["account_type", "account_status", "currency"])
        df = cast_numeric(df, ["balance", "credit_limit"])

    elif name == "cards":
        df = standardize_strings(df, ["card_type", "card_status"])

    elif name == "merchants":
        df = standardize_strings(df, ["merchant_name", "merchant_category", "country", "city"])
        df = cast_numeric(df, ["risk_score"])

    elif name == "devices":
        df = standardize_strings(df, ["device_type", "os_type"])

    elif name == "transactions":
        df = cast_numeric(df, ["amount"])
        df = df.withColumn("transaction_date",
                           F.to_timestamp(F.col("transaction_date"))) if "transaction_date" in df.columns else df

    return add_silver_audit_cols(df)

print("Transformation library registered.")

Transformation library registered.


In [0]:
# ============================================================
# CELL 7: Main Silver transformation loop — metadata-driven
# ============================================================
results = []

for row in metadata_rows:

    table_start  = datetime.utcnow()
    source_full  = f"{CATALOG}.{row.source_table}"
    target_full  = f"{CATALOG}.{row.target_table}"
    source_short = row.source_table.split(".")[-1]

    print(f"\n{'='*60}")
    print(f"[START] {source_short} -> {row.target_table}")
    print(f"  Strategy : {row.transformation_type}")
    print(f"{'='*60}")

    try:
        # ── 1. Read from Bronze ──────────────────────────────
        bronze_df = spark.table(source_full)
        total_rows = bronze_df.count()
        print(f"  [READ] {total_rows:,} rows from {source_full}")

        # ── 2. Apply transformations ─────────────────────────
        transformed_df = apply_transformation(bronze_df, row.source_table)

        # ── 3. Remove duplicates ─────────────────────────────
        deduped_df, dup_count = remove_duplicates(transformed_df, source_short, row.dq_rules)

        # ── 4. Run DQ engine ─────────────────────────────────
        clean_df, reject_count = run_dq_engine(deduped_df, source_short, row.dq_rules)

        passed_rows = clean_df.count()

        # ── 5. Write to Silver ───────────────────────────────
        if row.transformation_type == "SCD2":
            # Determine key column from dedup_key in rules
            rules   = json.loads(row.dq_rules) if row.dq_rules else {}
            key_col = rules.get("dedup_key", "id").split(",")[0].strip()
            apply_scd2(spark, clean_df, target_full, key_col)
        else:
            (
                clean_df.write
                    .format("delta")
                    .mode(row.load_strategy.lower())
                    .option("mergeSchema", "true")
                    .saveAsTable(target_full)
            )
            print(f"  [WRITE] {passed_rows:,} rows -> {target_full}")

        # ── 6. Write DQ metrics ──────────────────────────────
        write_dq_metrics(row.target_table, total_rows, passed_rows, reject_count, dup_count)

        results.append({
            "table":    row.target_table,
            "status":   "SUCCESS",
            "total":    total_rows,
            "passed":   passed_rows,
            "rejected": reject_count,
            "dupes":    dup_count
        })

    except Exception as e:
        err = traceback.format_exc()[-2000:]
        print(f"  [FAIL] {row.target_table}: {e}")
        write_dq_metrics(row.target_table, 0, 0, 0, 0)
        results.append({"table": row.target_table, "status": "FAILED", "total": 0,
                        "passed": 0, "rejected": 0, "dupes": 0, "error": err})

/home/spark-e24d9109-d362-4fd7-bdd2-8f/.ipykernel/2061/command-6388236773342283-1717844053:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  table_start  = datetime.utcnow()



[START] brz_customers -> silver.silver_customers
  Strategy : SCD2
  [READ] 30,000 rows from ubuntu_bank_fraud.bronze.brz_customers
  [DEDUP] brz_customers: 0 duplicates removed
  [DQ] brz_customers: 30000 total, 0 rejected, 30000 clean


/home/spark-e24d9109-d362-4fd7-bdd2-8f/.ipykernel/2061/command-6388236773342281-560589567:11: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()


  [SCD2] ubuntu_bank_fraud.silver.silver_customers: 30000 new/changed versions inserted


/home/spark-e24d9109-d362-4fd7-bdd2-8f/.ipykernel/2061/command-6388236773342279-4290338126:19: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  row = [(table_name, int(total), int(passed), int(rejected), int(duplicated), pass_rate, BATCH_ID, datetime.utcnow())]
/home/spark-e24d9109-d362-4fd7-bdd2-8f/.ipykernel/2061/command-6388236773342283-1717844053:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  table_start  = datetime.utcnow()


...


[START] brz_cards -> silver.silver_cards
  Strategy : OVERWRITE
  [READ] 39,705 rows from ubuntu_bank_fraud.bronze.brz_cards
  [DEDUP] brz_cards: 0 duplicates removed
  [DQ] brz_cards: 39705 total, 0 rejected, 39705 clean
  [WRITE] 39,705 rows -> ubuntu_bank_fraud.silver.silver_cards

[START] brz_merchants -> silver.silver_merchants
  Strategy : OVERWRITE
  [READ] 2,200 rows from ubuntu_bank_fraud.bronze.brz_merchants
  [DEDUP] brz_merchants: 0 duplicates removed
  [DQ] brz_merchants: 2200 total, 0 rejected, 2200 clean
  [WRITE] 2,200 rows -> ubuntu_bank_fraud.silver.silver_merchants

[START] brz_devices -> silver.silver_devices
  Strategy : SCD2
  [READ] 28,000 rows from ubuntu_bank_fraud.bronze.brz_devices
  [DEDUP] brz_devices: 0 duplicates removed
  [DQ] brz_devices: 28000 total, 0 rejected, 28000 clean


/home/spark-e24d9109-d362-4fd7-bdd2-8f/.ipykernel/2061/command-6388236773342281-560589567:11: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()


  [SCD2] ubuntu_bank_fraud.silver.silver_devices: 28000 new/changed versions inserted


/home/spark-e24d9109-d362-4fd7-bdd2-8f/.ipykernel/2061/command-6388236773342279-4290338126:19: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  row = [(table_name, int(total), int(passed), int(rejected), int(duplicated), pass_rate, BATCH_ID, datetime.utcnow())]
/home/spark-e24d9109-d362-4fd7-bdd2-8f/.ipykernel/2061/command-6388236773342283-1717844053:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  table_start  = datetime.utcnow()



[START] brz_customer_device_link -> silver.silver_customer_device_link
  Strategy : OVERWRITE
  [READ] 34,706 rows from ubuntu_bank_fraud.bronze.brz_customer_device_link
  [DEDUP] brz_customer_device_link: 0 duplicates removed
  [DQ] brz_customer_device_link: 34706 total, 0 rejected, 34706 clean
  [WRITE] 34,706 rows -> ubuntu_bank_fraud.silver.silver_customer_device_link

[START] brz_transactions -> silver.silver_transaction_fact
  Strategy : OVERWRITE
  [READ] 2,200,000 rows from ubuntu_bank_fraud.bronze.brz_transactions
  [DEDUP] brz_transactions: 0 duplicates removed
  [DQ] brz_transactions: 2200000 total, 0 rejected, 2200000 clean
  [WRITE] 2,200,000 rows -> ubuntu_bank_fraud.silver.silver_transaction_fact


In [0]:
# ============================================================
# CELL 8: Silver Feature Store — silver_transaction_features
# ============================================================
print("\nBuilding silver_transaction_features...")

txn_df = spark.table(f"{CATALOG}.silver.silver_transaction_fact")

# ── Ensure transaction_date is timestamp ─────────────────────
txn_df = txn_df.withColumn(
    "transaction_date",
    F.to_timestamp(F.col("transaction_date"))
)

# ── Alias the actual column names to standard names ──────────
# transaction_amount -> amount, source_account_id -> account_id
txn_df = (
    txn_df
    .withColumn("amount",     F.col("transaction_amount"))
    .withColumn("account_id", F.col("source_account_id"))
)

# ── TIME FEATURES ────────────────────────────────────────────
txn_df = (
    txn_df
    .withColumn("hour_of_day",  F.hour("transaction_date"))
    .withColumn("day_of_week",  F.dayofweek("transaction_date"))
    .withColumn("week_of_year", F.weekofyear("transaction_date"))
    .withColumn("month_number", F.month("transaction_date"))
    .withColumn("is_weekend",   (F.dayofweek("transaction_date").isin([1, 7])).cast(IntegerType()))
    .withColumn("is_month_end", (F.dayofmonth(F.last_day("transaction_date")) ==
                                 F.dayofmonth("transaction_date")).cast(IntegerType()))
    .withColumn("off_hours_flag", (F.hour("transaction_date").between(0, 5)).cast(IntegerType()))
)

# ── ENRICH WITH CUSTOMER_ID VIA ACCOUNTS ────────────────────
accounts_bridge = (
    spark.table(f"{CATALOG}.silver.silver_accounts")
    .select("account_id", "customer_id")
    .dropDuplicates(["account_id"])
)

txn_df = txn_df.join(accounts_bridge, "account_id", "left")
print(f"  [INFO] customer_id joined via silver_accounts")

# ── CUSTOMER VELOCITY FEATURES ───────────────────────────────
DAY1  = 86400
DAY7  = 86400 * 7
DAY30 = 86400 * 30

w_cust = Window.partitionBy("customer_id").orderBy(F.col("transaction_date").cast("long"))

txn_df = (
    txn_df
    .withColumn("customer_txn_count_1d",
        F.count("transaction_id").over(w_cust.rangeBetween(-DAY1, 0)))
    .withColumn("customer_txn_count_7d",
        F.count("transaction_id").over(w_cust.rangeBetween(-DAY7, 0)))
    .withColumn("customer_txn_count_30d",
        F.count("transaction_id").over(w_cust.rangeBetween(-DAY30, 0)))
    .withColumn("customer_amount_sum_1d",
        F.sum("amount").over(w_cust.rangeBetween(-DAY1, 0)))
    .withColumn("customer_amount_sum_7d",
        F.sum("amount").over(w_cust.rangeBetween(-DAY7, 0)))
    .withColumn("customer_amount_sum_30d",
        F.sum("amount").over(w_cust.rangeBetween(-DAY30, 0)))
    .withColumn("avg_customer_amount_7d",
        F.avg("amount").over(w_cust.rangeBetween(-DAY7, 0)))
    .withColumn("avg_customer_amount_30d",
        F.avg("amount").over(w_cust.rangeBetween(-DAY30, 0)))
    .withColumn("amount_vs_customer_avg",
        F.col("amount") / F.when(
            F.col("avg_customer_amount_7d").isNull() | (F.col("avg_customer_amount_7d") == 0),
            F.lit(1)
        ).otherwise(F.col("avg_customer_amount_7d")))
)

# ── MERCHANT FEATURES ────────────────────────────────────────
w_merch = Window.partitionBy("merchant_id").orderBy(F.col("transaction_date").cast("long"))

txn_df = (
    txn_df
    .withColumn("merchant_txn_count_1d",
        F.count("transaction_id").over(w_merch.rangeBetween(-DAY1, 0)))
    .withColumn("merchant_txn_count_7d",
        F.count("transaction_id").over(w_merch.rangeBetween(-DAY7, 0)))
    .withColumn("merchant_volume_7d",
        F.sum("amount").over(w_merch.rangeBetween(-DAY7, 0)))
    .withColumn("merchant_customer_count",
        F.approx_count_distinct("customer_id").over(
            Window.partitionBy("merchant_id")))
)

# ── DEVICE FEATURES ──────────────────────────────────────────
device_link_df = spark.table(f"{CATALOG}.silver.silver_customer_device_link")

device_stats = (
    device_link_df
    .groupBy("device_id")
    .agg(F.countDistinct("customer_id").alias("device_customer_count"))
)

account_device_stats = (
    device_link_df
    .join(
        spark.table(f"{CATALOG}.silver.silver_accounts")
            .select("account_id", "customer_id")
            .dropDuplicates(["account_id"]),
        "customer_id", "left"
    )
    .groupBy("customer_id")
    .agg(F.countDistinct("account_id").alias("device_account_count"))
)

# device_id already exists on transactions — use it directly
txn_df = txn_df.join(device_stats,         "device_id",   "left")
txn_df = txn_df.join(account_device_stats,  "customer_id", "left")

# ── FRAUD INDICATOR FLAGS ────────────────────────────────────
txn_features = (
    txn_df
    .withColumn("high_velocity_flag",
        (F.col("customer_txn_count_1d") > 10).cast(IntegerType()))
    .withColumn("shared_device_flag",
        (F.col("device_customer_count") > 1).cast(IntegerType()))
    .withColumn("high_risk_amount_flag",
        (F.col("amount_vs_customer_avg") > 3).cast(IntegerType()))
    .withColumn("new_merchant_flag",
        (F.col("merchant_txn_count_7d") < 5).cast(IntegerType()))
    .withColumn("_feature_ts",    F.current_timestamp())
    .withColumn("_feature_batch", F.lit(BATCH_ID))
)

# ── WRITE FEATURE STORE ──────────────────────────────────────
(
    txn_features.write
        .format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .saveAsTable(f"{CATALOG}.silver.silver_transaction_features")
)

feat_count = spark.table(f"{CATALOG}.silver.silver_transaction_features").count()
print(f"  [FEATURES] silver_transaction_features: {feat_count:,} rows written")

write_dq_metrics("silver.silver_transaction_features", feat_count, feat_count, 0, 0)


Building silver_transaction_features...
  [INFO] customer_id joined via silver_accounts
  [FEATURES] silver_transaction_features: 2,200,000 rows written


/home/spark-e24d9109-d362-4fd7-bdd2-8f/.ipykernel/2061/command-6388236773342279-4290338126:19: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  row = [(table_name, int(total), int(passed), int(rejected), int(duplicated), pass_rate, BATCH_ID, datetime.utcnow())]
